# Interactive play (mode 1)

Have a live conversation with the Gemma 4 E4B agent over **one persistent game**.
Each turn you see the *exact* frame the agent saw, its reply, and -- if it made a
move -- the board after the move. The **Restart conversation** button re-initializes
the env (a brand new bare level) and starts a fresh conversation thread.

Prereqs:
- Neo4j is up (`bash scripts/vast_neo4j_launch.sh`, or `docker compose up -d neo4j`).
- The semantic model is seeded once (`python -m agent.runner seed`).
- `pip install -r requirements.txt` (installs `ipywidgets`).

The first cell connects NAMS and loads Gemma, so it can take a minute.

In [ ]:
import os
import sys

# Run from the repo root so `agent` imports and `.env` resolve.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from agent.interactive import InteractiveSession

# Connects NAMS on a background loop and loads Gemma 4 E4B (first run is slow).
session = InteractiveSession()
print("session_id:", session.session_id)

In [ ]:
import ipywidgets as widgets
from IPython.display import Image, clear_output, display

question_box = widgets.Textarea(
    value="",
    placeholder="Ask the agent something, or tell it to make a move...",
    description="You:",
    layout=widgets.Layout(width="600px", height="70px"),
)
ask_btn = widgets.Button(description="Ask", button_style="primary")
restart_btn = widgets.Button(description="Restart conversation", button_style="warning")
out = widgets.Output()

# Rendered frames are 224x224 native; upscale on display to about half a normal
# cell so the agent and gold are clearly visible on a good monitor.
FRAME_WIDTH = 420  # px


def _show_frame(path, caption):
    if path:
        print(caption)
        display(Image(filename=path, width=FRAME_WIDTH))


def on_ask(_):
    q = question_box.value.strip()
    if not q:
        return
    ask_btn.disabled = True
    try:
        with out:
            print(f"\n=== You: {q}")
            result = session.ask(q)
            _show_frame(result["before_path"], "-- frame the agent saw --")
            print(f"Agent: {result['raw']}")
            if result["action"]:
                print(
                    f"[move applied: {result['action']}  "
                    f"gold_collected={result['gold_collected']}  "
                    f"gold_remaining={result['gold_remaining']}]"
                )
                _show_frame(result["after_path"], "-- frame after the move --")
            else:
                print("[no move: answer only]")
        question_box.value = ""
    finally:
        ask_btn.disabled = False


def on_restart(_):
    info = session.restart()
    with out:
        clear_output()
        print(f"New conversation. session_id={info['session_id']}")
        _show_frame(info["frame_path"], "-- fresh board --")


ask_btn.on_click(on_ask)
restart_btn.on_click(on_restart)

display(widgets.VBox([question_box, widgets.HBox([ask_btn, restart_btn]), out]))

# Show the current starting board.
with out:
    print(f"session_id={session.session_id}")
    _show_frame(session.current_frame_path(), "-- starting board --")

When you're done, release the model and close the memory client:

In [ ]:
session.close()